# SQL + Pandas Comparison Notebook
### Assignment: Week 3, Day 14 — SQL Queries with Pandas Equivalents

## Setup: Create Sample DataFrames (same data as our SQL tables)

In [ ]:
import pandas as pd

# ---------- departments ----------
dept_df = pd.DataFrame({
    'dept_id':   [1, 2, 3, 4],
    'dept_name': ['Engineering', 'HR', 'Sales', 'Marketing'],
    'budget':    [500000, 200000, 300000, 250000]
})

# ---------- employees ----------
df = pd.DataFrame({
    'emp_id':    [1,2,3,4,5,6,7,8,9,10],
    'name':      ['Alice','Bob','Carol','David','Eve','Frank','Grace','Hank','Ivy','Jack'],
    'salary':    [90000,75000,60000,55000,80000,70000,65000,72000,58000,95000],
    'dept_id':   [1,1,2,2,3,3,4,1,4,1],
    'hire_date': pd.to_datetime(['2020-01-15','2019-06-01','2021-03-10','2022-07-20',
                                 '2018-11-05','2020-09-15','2021-01-01','2023-02-28',
                                 '2022-05-14','2017-08-19'])
})

# ---------- projects ----------
proj_df = pd.DataFrame({
    'project_id':   [1,2,3,4,5],
    'project_name': ['Website Revamp','HR System','Sales Dashboard','Ad Campaign','Data Pipeline'],
    'lead_emp_id':  [1,3,5,7,10],
    'budget':       [120000,80000,150000,90000,200000],
    'start_date':   pd.to_datetime(['2024-01-01','2024-02-01','2024-03-01','2024-04-01','2024-05-01']),
    'end_date':     pd.to_datetime(['2024-06-30','2024-08-31','2024-09-30','2024-10-31','2024-12-31'])
})

print('Tables ready!')
print('Employees:', df.shape)
print('Departments:', dept_df.shape)
print('Projects:', proj_df.shape)

---
## Part A — 5 Basic SELECT / WHERE Queries

In [ ]:
# Q1: Get all employees
# SQL: SELECT * FROM employees;

df

In [ ]:
# Q2: Employees with salary > 70000
# SQL: SELECT * FROM employees WHERE salary > 70000;

df[df['salary'] > 70000]

In [ ]:
# Q3: Employees in Engineering (dept_id = 1)
# SQL: SELECT * FROM employees WHERE dept_id = 1;

df[df['dept_id'] == 1]

In [ ]:
# Q4: Employees hired after 2020
# SQL: SELECT name, hire_date FROM employees WHERE hire_date > '2020-01-01';

df[df['hire_date'] > '2020-01-01'][['name', 'hire_date']]

In [ ]:
# Q5: Employees with salary between 60000 and 80000
# SQL: SELECT name, salary FROM employees WHERE salary BETWEEN 60000 AND 80000;

df[(df['salary'] >= 60000) & (df['salary'] <= 80000)][['name', 'salary']]

---
## Part A — 5 JOIN Queries

In [ ]:
# Q6: Employee name + department name
# SQL: SELECT e.name, d.dept_name FROM employees e JOIN departments d ON e.dept_id = d.dept_id;

df.merge(dept_df, on='dept_id')[['name', 'dept_name']]

In [ ]:
# Q7: Employees earning > 70000 with their department
# SQL: SELECT e.name, e.salary, d.dept_name FROM employees e
#      JOIN departments d ON e.dept_id = d.dept_id WHERE e.salary > 70000;

merged = df.merge(dept_df, on='dept_id')
merged[merged['salary'] > 70000][['name', 'salary', 'dept_name']]

In [ ]:
# Q8: All departments even if no employees (LEFT JOIN)
# SQL: SELECT d.dept_name, e.name FROM departments d LEFT JOIN employees e ON d.dept_id = e.dept_id;

dept_df.merge(df, on='dept_id', how='left')[['dept_name', 'name']]

In [ ]:
# Q9: Employees hired before 2021 with department name
# SQL: SELECT e.name, e.hire_date, d.dept_name FROM employees e
#      JOIN departments d ON e.dept_id = d.dept_id WHERE e.hire_date < '2021-01-01';

merged = df.merge(dept_df, on='dept_id')
merged[merged['hire_date'] < '2021-01-01'][['name', 'hire_date', 'dept_name']]

In [ ]:
# Q10: Employee name + department name + department budget
# SQL: SELECT e.name, d.dept_name, d.budget FROM employees e
#      JOIN departments d ON e.dept_id = d.dept_id;

df.merge(dept_df, on='dept_id')[['name', 'dept_name', 'budget']]

---
## Part A — 5 Aggregation Queries

In [ ]:
# Q11: Average salary per department
# SQL: SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id;

df.groupby('dept_id')['salary'].mean().reset_index(name='avg_salary')

In [ ]:
# Q12: Count of employees per department
# SQL: SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id;

df.groupby('dept_id')['emp_id'].count().reset_index(name='employee_count')

In [ ]:
# Q13: Maximum salary per department
# SQL: SELECT dept_id, MAX(salary) FROM employees GROUP BY dept_id;

df.groupby('dept_id')['salary'].max().reset_index(name='max_salary')

In [ ]:
# Q14: Departments where avg salary > 70000  (equivalent of HAVING)
# SQL: SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id HAVING AVG(salary) > 70000;

result = df.groupby('dept_id')['salary'].mean().reset_index(name='avg_salary')
result[result['avg_salary'] > 70000]

In [ ]:
# Q15: Total salary per department
# SQL: SELECT dept_id, SUM(salary) FROM employees GROUP BY dept_id;

df.groupby('dept_id')['salary'].sum().reset_index(name='total_salary')

---
## Part A — 5 Advanced SQL Problems (Pandas Equivalents)

In [ ]:
# A1: Running Total (equivalent of SUM OVER ORDER BY)
# SQL: SELECT name, salary, SUM(salary) OVER (ORDER BY emp_id) AS running_total FROM employees;

result = df[['name','salary']].copy()
result['running_total'] = df['salary'].cumsum()
result

In [ ]:
# A2: Top 3 earners per department (equivalent of RANK OVER PARTITION BY)
# SQL: RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC)

df['rank'] = df.groupby('dept_id')['salary'].rank(ascending=False, method='min').astype(int)
df[df['rank'] <= 3][['name','dept_id','salary','rank']].sort_values(['dept_id','rank'])

In [ ]:
# A3: Month-over-Month growth (equivalent of LAG)
# SQL: LAG(revenue) OVER (ORDER BY month)

monthly = pd.DataFrame({
    'month':   pd.to_datetime(['2024-01-01','2024-02-01','2024-03-01']),
    'revenue': [100000, 120000, 110000]
})
monthly['prev_month_revenue'] = monthly['revenue'].shift(1)       # LAG
monthly['growth'] = monthly['revenue'] - monthly['prev_month_revenue']
monthly

In [ ]:
# A4: CTE equivalent — employees earning above their department average
# SQL: WITH dept_avg AS (...) SELECT ... WHERE e.salary > da.avg_sal;

dept_avg = df.groupby('dept_id')['salary'].mean().reset_index(name='avg_sal')  # This is the CTE
merged = df.merge(dept_avg, on='dept_id')
merged[merged['salary'] > merged['avg_sal']][['name','salary','avg_sal']]

In [ ]:
# A5: Correlated Subquery — employees earning above their dept average
# SQL: WHERE salary > (SELECT AVG(salary) FROM employees WHERE dept_id = e.dept_id)

dept_avg = df.groupby('dept_id')['salary'].mean()              # dept averages
df['dept_avg'] = df['dept_id'].map(dept_avg)                   # map back to each row
df[df['salary'] > df['dept_avg']][['name','dept_id','salary','dept_avg']]

---
## Part B — Projects Table (Pandas)

In [ ]:
# B1: 3-table JOIN — employee name + dept budget + project budget
# SQL: SELECT e.name, d.dept_name, d.budget, p.project_name, p.budget FROM projects p
#      JOIN employees e ON p.lead_emp_id = e.emp_id JOIN departments d ON e.dept_id = d.dept_id;

step1 = proj_df.merge(df[['emp_id','name','dept_id']], left_on='lead_emp_id', right_on='emp_id')
step2 = step1.merge(dept_df, on='dept_id')
step2[['name','dept_name','budget_y','project_name','budget_x']].rename(
    columns={'budget_x':'project_budget','budget_y':'dept_budget'})

In [ ]:
# B2: Departments where total project budget > department budget
# SQL: HAVING SUM(p.budget) > d.budget

step1 = proj_df.merge(df[['emp_id','dept_id']], left_on='lead_emp_id', right_on='emp_id')
step2 = step1.merge(dept_df, on='dept_id')
grouped = step2.groupby(['dept_name','budget_y'])['budget_x'].sum().reset_index()
grouped.columns = ['dept_name','dept_budget','total_project_budget']
grouped[grouped['total_project_budget'] > grouped['dept_budget']]

---
## Part C — Interview Questions

### C1 — SQL Logical Execution Order

SQL runs in this order (NOT the order you write it):

| Step | Clause     | What it does                     |
|------|------------|----------------------------------|
| 1    | FROM       | Which table to read              |
| 2    | JOIN       | Combine tables                   |
| 3    | WHERE      | Filter individual rows           |
| 4    | GROUP BY   | Group rows together              |
| 5    | HAVING     | Filter groups                    |
| 6    | SELECT     | Pick which columns to show       |
| 7    | ORDER BY   | Sort the result                  |
| 8    | LIMIT      | How many rows to return          |

**Why aliases matter:** `SELECT` runs at step 6, AFTER `WHERE` (step 3) and `GROUP BY` (step 4).  
So you **cannot use a SELECT alias inside WHERE or GROUP BY** — SQL hasn't created the alias yet!

In [ ]:
# C2: Employees earning above company average, showing their dept average salary
# (No subqueries/CTEs in SQL — we use a self-join)

company_avg = df['salary'].mean()
dept_avg    = df.groupby('dept_id')['salary'].mean()
df['dept_avg_salary'] = df['dept_id'].map(dept_avg)

result = df[df['salary'] > company_avg][['name','salary','dept_avg_salary']]
print(f'Company average: {company_avg}')
result

In [ ]:
# C3: Debugged Query
# BUG: WHERE AVG(salary) > 70000  <-- Cannot use aggregate function in WHERE
# FIX: Use HAVING after GROUP BY

# Pandas equivalent of the FIXED version:
result = df.groupby('dept_id')['salary'].mean().reset_index(name='avg_sal')
result[result['avg_sal'] > 70000]

---
## Part D — AI-Generated Interview Questions & Answers

In [ ]:
# D1: Employees with NO projects assigned (LEFT JOIN + NULL check)
merged = df.merge(proj_df, left_on='emp_id', right_on='lead_emp_id', how='left')
merged[merged['project_id'].isna()][['name']]

In [ ]:
# D2: NULL salary handling — salary is NULL or below 50000
df[df['salary'].isna() | (df['salary'] < 50000)]

In [ ]:
# D3: Rank employees by salary within department (Window function)
df['salary_rank'] = df.groupby('dept_id')['salary'].rank(ascending=False, method='min').astype(int)
df[['name','dept_id','salary','salary_rank']].sort_values(['dept_id','salary_rank'])

In [ ]:
# D4: Departments where max salary > double the min salary
result = df.groupby('dept_id')['salary'].agg(['max','min']).reset_index()
result[result['max'] > 2 * result['min']]

### Evaluation of AI-Generated Questions

| # | Question Topic | Difficulty | Complete Answer? |
|---|---------------|------------|------------------|
| 1 | JOINs + NULL  | Medium ✅  | Yes ✅           |
| 2 | NULL handling | Easy-Medium| Yes ✅           |
| 3 | Performance / EXPLAIN | Medium ✅ | Yes ✅     |
| 4 | Window functions (RANK) | Medium ✅ | Yes ✅  |
| 5 | Aggregation + HAVING | Medium ✅ | Yes ✅    |

**Overall:** Questions are genuinely medium-difficulty. They cover real interview topics.  
Answers are complete and produce correct results when run on our employees database.